In [1]:
print("Let's Start")

Let's Start


In [ ]:
import retry:
    dagshub.init(repo_owner='jubito27', repo_name='mlops-mini-project', mlflow=True)
    mlflow.set_tracking_uri('https://dagshub.com/jubito27/mlops-mini-project.mlflow')
except Exception as e:
    print("Error occured as :" , e)
mlflow.set_experiment('BOW vs TFIDF')
import string
import automac
import mlflow
import mlflow.sklearn
import dagshub
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score , precision_score , recall_score , f1_score 
from sklearn.feature_extraction.text import CountVectorizer , TfidfVectorizer

In [6]:
import xgboost
import lightgbm
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB

In [15]:
df = pd.read_csv('/mnt/WindowsD/Artificial Intelligence/Datasets/tweets_emotions.csv' , index_col=0).drop(columns='tweet_id')
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [16]:
x=(df['sentiment']=='happiness') | (df['sentiment'] == 'sadness')
df = df[x]
df['sentiment'] =df['sentiment'].replace({'sadness' : 0 , 'happiness' : 1}) 
x_ = df['content'].apply(automac.text.transform)
y = df['sentiment']
y = pd.to_numeric(y, errors='raise').astype(int)


In [14]:
x_

Series([], Name: content, dtype: str)

In [25]:
try:
    dagshub.init(repo_owner='jubito27', repo_name='mlops-mini-project', mlflow=True)
    mlflow.set_tracking_uri('https://dagshub.com/jubito27/mlops-mini-project.mlflow')
except Exception as e:
    print("Error occured as :" , e)
mlflow.set_experiment('BOW vs TFIDF')

Accessing as jubito27

Initialized MLflow to track repo "jubito27/mlops-mini-project"

Repository jubito27/mlops-mini-project initialized!

2026/11/14 11:41:08 INFO mlflow.tracking.fluent: Experiment with name 'BOW vs TFIDF' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/351ebfaab8754899833e3d8d97234731', creation_time=1794636669908, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1794636669908, lifecycle_stage='active', name='BOW vs TFIDF', tags={}, trace_location=None, workspace='default'>

In [30]:
vectorizers = {
    'bow' : CountVectorizer(max_features=100),
    'tfidf' : TfidfVectorizer(max_features=100)

}

algorithms = {
    'LogisticRegression' : LogisticRegression(),
    'RandomForest' : RandomForestClassifier(n_estimators=50),
    'NaiveBayes' : MultinomialNB(),
    'Xgboost' : xgboost.XGBClassifier(n_estimators=50),
    'Lightgbm' : lightgbm.LGBMClassifier(n_estimators=50)
}

In [33]:
from sklearn.tree import _tree

with mlflow.start_run(run_name = 'All Experiments') as parent:
    for algo_name , algorithm in algorithms.items():
        for vect , vectorizer in vectorizers.items():
            with mlflow.start_run(run_name=f"{algo_name} with {vect}" , nested=True) as child:
                x = vectorizer.fit_transform(x_).astype(np.float32)

                x_train , x_test , y_train , y_test= train_test_split(x , y , test_size=0.2 , random_state=0)

                mlflow.log_param("vectorizer" , vect)
                mlflow.log_param("algorithm" , algo_name)
                mlflow.log_param("test_size" , 0.2)

                model = algorithm
                model.fit(x_train , y_train)

                if algo_name == 'LogisticRegression':
                    mlflow.log_param("C", model.C)
                elif algo_name == 'MultinomialNB':
                    mlflow.log_param("alpha", model.alpha)
                elif algo_name == 'XGBoost':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("learning_rate", model.learning_rate)
                elif algo_name == 'RandomForest':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("max_depth", model.max_depth)
                elif algo_name == 'GradientBoosting':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("learning_rate", model.learning_rate)


                y_pred = model.predict(x_test)
                accuracy = accuracy_score(y_test, y_pred)
                precision = precision_score(y_test, y_pred)
                recall = recall_score(y_test, y_pred)
                f1 = f1_score(y_test, y_pred)

                
                # Log evaluation metrics
                mlflow.log_metric("accuracy", accuracy)
                mlflow.log_metric("precision", precision)
                mlflow.log_metric("recall", recall)
                mlflow.log_metric("f1_score", f1)


                mlflow.sklearn.log_model(
                    model,
                    artifact_path="model",
                    skops_trusted_types=[
                        "sklearn.tree._tree.Tree",
                        "xgboost.core.Booster",
                        "xgboost.sklearn.XGBClassifier",
                        "collections.OrderedDict",
                        "lightgbm.basic.Booster",
                        "lightgbm.sklearn.LGBMClassifier",
                    ],
                )
            

                # Save and log the notebook
                mlflow.log_artifact("2_bowVStfidf.ipynb")
                # Print the results for verification
                print(f"Algorithm: {algo_name}, Feature Engineering: {vect}")
                print(f"Accuracy: {accuracy}")
                print(f"Precision: {precision}")
                print(f"Recall: {recall}")
                print(f"F1 Score: {f1}")


2026/11/14 13:47:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/14 13:47:50 WARNING mlflow.utils.requirements_utils: Found torch version (2.13.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.13.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Algorithm: LogisticRegression, Feature Engineering: bow
Accuracy: 0.7286746987951808
Precision: 0.7660753880266076
Recall: 0.6625119846596357
F1 Score: 0.7105398457583547
🏃 View run LogisticRegression with bow at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/2/runs/8d7f78ed34ff4c7dafc534acfd95313a
🧪 View experiment at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/2


2026/11/14 13:48:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/14 13:48:41 WARNING mlflow.utils.requirements_utils: Found torch version (2.13.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.13.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Algorithm: LogisticRegression, Feature Engineering: tfidf
Accuracy: 0.7262650602409638
Precision: 0.7573131094257854
Recall: 0.6701821668264621
F1 Score: 0.711088504577823
🏃 View run LogisticRegression with tfidf at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/2/runs/5da413af3ab2418db488d894a3a7aa4b
🧪 View experiment at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/2


2026/11/14 13:48:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/14 13:49:22 WARNING mlflow.utils.requirements_utils: Found torch version (2.13.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.13.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Algorithm: RandomForest, Feature Engineering: bow
Accuracy: 0.7016867469879519
Precision: 0.7319474835886215
Recall: 0.6414189837008629
F1 Score: 0.683699540112417
🏃 View run RandomForest with bow at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/2/runs/a3f4187b63da416a808c55aabaaa40d5
🧪 View experiment at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/2


... (10 outputs left out)

2026/11/14 14:34:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/14 14:35:22 WARNING mlflow.utils.requirements_utils: Found torch version (2.13.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.13.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Algorithm: Lightgbm, Feature Engineering: bow
Accuracy: 0.727710843373494
Precision: 0.7655555555555555
Recall: 0.660594439117929
F1 Score: 0.7092125579001544
🏃 View run Lightgbm with bow at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/2/runs/ec73e91d578a44458fe5d658216fd17a
🧪 View experiment at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/2
[LightGBM] [Info] Number of positive: 4166, number of negative: 4133
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022150 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6574
[LightGBM] [Info] Number of data points in the train set: 8299, number of used features: 100
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.501988 -> initscore=0.007953
[LightGBM] [Info] Start training from score 0.007953


2026/11/14 14:35:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/14 14:35:56 WARNING mlflow.utils.requirements_utils: Found torch version (2.13.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.13.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Algorithm: Lightgbm, Feature Engineering: tfidf
Accuracy: 0.7253012048192771
Precision: 0.7573449401523396
Recall: 0.6673058485139022
F1 Score: 0.709480122324159
🏃 View run Lightgbm with tfidf at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/2/runs/2bf9e53ef3a8449d8e9a6f7f6dd192f9
🧪 View experiment at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/2
🏃 View run All Experiments at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/2/runs/87881ba45fa1460fb804d7b330f0612c
🧪 View experiment at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/2
